In [7]:
import pandas as pd
import numpy as np

In [8]:
df=pd.read_csv(r"D:\Bluestock\Project\Data\Raw\02_nav_history.csv")

In [9]:
df.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB


In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df["date"] = pd.to_datetime(df["date"])

In [13]:
df.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [14]:
df= df.sort_values(by=["amfi_code", "date"])

In [15]:
df.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [16]:
df.groupby(df["amfi_code"]).ngroups

40

In [17]:
df = df.set_index("date")

In [18]:
df.head()

,amfi_code,nav
date,,
2022-01-03,100016,520.4608
2022-01-04,100016,515.0971
2022-01-05,100016,521.7239
2022-01-06,100016,515.7880
2022-01-07,100016,515.1639


In [19]:
all_dates = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq='D'
)

In [20]:
groups = df.groupby("amfi_code")

In [21]:
processed_groups = []

In [26]:
for code, group in groups:

    all_dates = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq='D'
    )

    group = group.reindex(all_dates)

    group["amfi_code"] = code
    group["nav"] = group["nav"].ffill()

    group = group.reset_index()
    group.rename(columns={"index": "date"}, inplace=True)

    processed_groups.append(group)

In [27]:
clean_df = pd.concat(processed_groups, ignore_index=True)

In [28]:
clean_df.head()

,date,amfi_code,nav
0,2022-01-03,100016,520.4608
1,2022-01-04,100016,515.0971
2,2022-01-05,100016,521.7239
3,2022-01-06,100016,515.7880
4,2022-01-07,100016,515.1639


In [29]:
clean_df.isna().sum()

date         0
amfi_code    0
nav          0
dtype: int64

In [30]:

clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128640 entries, 0 to 128639
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   date       128640 non-null  datetime64[us]
 1   amfi_code  128640 non-null  int64         
 2   nav        128640 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 2.9 MB


In [33]:
clean_df[
    (clean_df["amfi_code"] == 119551) &
    (clean_df["date"] >= "2022-01-07") &
    (clean_df["date"] <= "2022-01-10")
].sort_values("date")

,date,amfi_code,nav
30556,2022-01-07,119551,55.3692
94876,2022-01-07,119551,55.3692
30557,2022-01-08,119551,55.3692
94877,2022-01-08,119551,55.3692
30558,2022-01-09,119551,55.3692
94878,2022-01-09,119551,55.3692
30559,2022-01-10,119551,55.2835
94879,2022-01-10,119551,55.2835


In [34]:
clean_df.duplicated().sum()

np.int64(64320)

In [38]:
clean_df.drop_duplicates(inplace=True)

In [41]:
t=clean_df["nav"]>0

In [44]:
t.value_counts()

nav
True    64320
Name: count, dtype: int64

In [45]:
clean_df["nav"].isna().sum()

np.int64(0)

In [ ]:
clean_df.to_csv("clean_nav_history.csv",index=False)